#Telecom Domain ReadOps Assignment
This notebook contains assignments to practice Spark read options and Databricks volumes. <br>
Sections: Sample data creation, Catalog & Volume creation, Copying data into Volumes, Path glob/recursive reads, toDF() column renaming variants, inferSchema/header/separator experiments, and exercises.<br>

![](https://fplogoimages.withfloats.com/actual/68009c3a43430aff8a30419d.png)
![](https://theciotimes.com/wp-content/uploads/2021/03/TELECOM1.jpg)

##First Import all required libraries & Create spark session object

##1. Write SQL statements to create:
1. A catalog named telecom_catalog_assign
2. A schema landing_zone
3. A volume landing_vol
4. Using dbutils.fs.mkdirs, create folders:<br>
/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/
/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/
/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/
5. Explain the difference between (Just google and understand why we are going for volume concept for prod ready systems):<br>
a. Volume vs DBFS/FileStore<br>
b. Why production teams prefer Volumes for regulated data<br>

In [0]:
%sql
create catalog if not exists telecom_catalog_assign;
create schema if not exists telecom_catalog_assign.landing_zone;
create volume if not exists telecom_catalog_assign.landing_zone.landing_vol;

In [0]:
dbutils.fs.mkdirs("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower")

##Data files to use in this usecase:
customer_csv = '''
101,Arun,31,Chennai,PREPAID
102,Meera,45,Bangalore,POSTPAID
103,Irfan,29,Hyderabad,PREPAID
104,Raj,52,Mumbai,POSTPAID
105,,27,Delhi,PREPAID
106,Sneha,abc,Pune,PREPAID
'''

usage_tsv = '''customer_id\tvoice_mins\tdata_mb\tsms_count
101\t320\t1500\t20
102\t120\t4000\t5
103\t540\t600\t52
104\t45\t200\t2
105\t0\t0\t0
'''

tower_logs_region1 = '''event_id|customer_id|tower_id|signal_strength|timestamp
5001|101|TWR01|-80|2025-01-10 10:21:54
5004|104|TWR05|-75|2025-01-10 11:01:12
'''

##2. Filesystem operations
1. Write code to copy the above datasets into your created Volume folders:
Customer → /Volumes/.../customer/
Usage → /Volumes/.../usage/
Tower (region-based) → /Volumes/.../tower/region1/ and /Volumes/.../tower/region2/

2. Write a command to validate whether files were successfully copied

In [0]:
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer")
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage")
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region1")
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region2")

In [0]:
customer_csv = ''' 
101,Arun,31,Chennai,PREPAID 
102,Meera,45,Bangalore,POSTPAID 
103,Irfan,29,Hyderabad,PREPAID 
104,Raj,52,Mumbai,POSTPAID 
105,,27,Delhi,PREPAID 
106,Sneha,abc,Pune,PREPAID '''

dbutils.fs.put("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer_raw.csv",customer_csv,overwrite=True)

df1 = spark.read.csv("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer_raw",sep=",",header=True,inferSchema=True).toDF("cust_id","cust_name","age","city","plan")
df1.show()


In [0]:
usage_tsv = '''customer_id\tvoice_mins\tdata_mb\tsms_count 
101\t320\t1500\t20 
102\t120\t4000\t5 
103\t540\t600\t52 
104\t45\t200\t2 
105\t0\t0\t0 '''

dbutils.fs.put("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage_raw.csv",usage_tsv,overwrite=True)

df2 = spark.read.csv("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage_raw.csv",sep="\t",header=True,inferSchema=True)
df2.show()


In [0]:
tower_logs_region1 = '''
event_id|customer_id|tower_id|signal_strength|timestamp 
5001|101|TWR01|-80|2025-01-10 10:21:54'''
dbutils.fs.put("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region1/tower_logs_raw.csv",tower_logs_region1,overwrite=True)

df3 = spark.read.csv("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region1/tower_logs_raw.csv",sep="|",header=True,inferSchema=True)
df3.show()

In [0]:
tower_logs_region2 = '''
event_id|customer_id|tower_id|signal_strength|timestamp 
5004|104|TWR05|-75|2025-01-10 11:01:12 '''
dbutils.fs.put("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region2/tower_logs_raw.csv",tower_logs_region2,overwrite=True)

df4 = spark.read.csv("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region2/tower_logs_raw.csv",sep="|",header=True,inferSchema=True)
df4.show()

##3. Directory Read Use Cases
1. Read all tower logs using:
Path glob filter (example: *.csv)
Multiple paths input
Recursive lookup

2. Demonstrate these 3 reads separately:
Using pathGlobFilter
Using list of paths in spark.read.csv([path1, path2])
Using .option("recursiveFileLookup","true")

3. Compare the outputs and understand when each should be used.

In [0]:
#Read all tower logs using: Path glob filter (example: *.csv) Multiple paths input Recursive lookup
df1 = spark.read.format("csv")\
    .option("header",True)\
    .option("inferSchema",True)\
    .option("recursiveFileLookup",True)\
    .load(["/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region1/tower_logs_raw.csv",
           "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region2/tower_logs_raw.csv"] ,sep = "|")

df1.show()

In [0]:
#Demonstrate these 3 reads separately: Using pathGlobFilter Using list of paths in spark.read.csv([path1, path2]) Using .option("recursiveFileLookup","true")

print("Mulitple Path Read")
df1 = spark.read.format("csv")\
    .option("header",True)\
    .option("inferSchema",True)\
    .option("recursiveFileLookup",True)\
    .load(["/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region1/tower_logs_raw.csv",
           "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region2/tower_logs_raw.csv"] ,sep = "|")

df1.show()

print("Multi File in child folders under the same parent path")
df2 = spark.read.format("csv")\
    .option("header",True)\
    .option("inferSchema",True)\
    .option("recursiveFileLookup",True)\
    .load("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/reg*",sep = "|")

df2.show()

print("Multi File in child folders under the same parent path using pathGlobFilter")
df3 = spark.read.format("csv")\
    .option("header",True)\
    .option("inferSchema",True)\
    .option("recursiveFileLookup",True)\
    .option("pathGlobFilter","*.csv")\
    .load("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/" ,sep = "|")

df3.show()



##4. Schema Inference, Header, and Separator
1. Try the Customer, Usage files with the option and options using read.csv and format function:<br>
header=false, inferSchema=false<br>
or<br>
header=true, inferSchema=true<br>
2. Write a note on What changed when we use header or inferSchema  with true/false?<br>
3. How schema inference handled “abc” in age?<br>

In [0]:
#Try the Customer, Usage files with the option and options using read.csv and format function:
#header=false, inferSchema=false or header=true, inferSchema=true

df = spark.read.format("csv")\
    .option("header",True)\
    .option("inferSchema",True)\
    .load("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer_raw.csv")
print("header,inferschema = True")
df.show()


df1 = spark.read.format("csv")\
    .option("header",False)\
    .option("inferSchema",False)\
    .load("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer_raw.csv")
print("header,inferschema = False")
df1.show()

######Write a note on What changed when we use header or inferSchema with true/false?
  As I already loaded both files with inference property enabled, all columns are already converted to string data type and not facing any issues.

######How schema inference handled “abc” in age?
  Age column is conidered as "string" while using inference property.
  


##5. Column Renaming Usecases
1. Apply column names using string using toDF function for customer data
2. Apply column names and datatype using the schema function for usage data
3. Apply column names and datatype using the StructType with IntegerType, StringType, TimestampType and other classes for towers data 

In [0]:
#Apply column names using string using toDF function for customer data
df = spark.read.csv("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer_raw.csv").toDF("cust_id","cust_name","age","location","plan")
df.show()

In [0]:
#Apply column names and datatype using the schema function for usage data

struct_schema = "cust_id int, cust_name string, age int, location string, plan string"
df = spark.read.schema(struct_schema).csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer_raw.csv",sep=',',inferSchema=True)
df.show()

In [0]:
#Apply column names and datatype using the StructType with IntegerType, StringType, TimestampType and other classes for towers data

from pyspark.sql.types import StructType,StructField,StringType,IntegerType,TimestampType

cust_schema = StructType([StructField("event_id",IntegerType(),True),
                      StructField("cust_id",StringType(),True),
                      StructField("tower_id",IntegerType(),True),
                      StructField("signal_strength",IntegerType(),True),
                      StructField("timestamp",TimestampType(),True)])
df = spark.read.schema(cust_schema)\
    .format("csv")\
    .option("pathGlobFilter","*.csv")\
    .option("header","True")\
    .option("recursiveFileLookup","True")\
    .option("delimiter","|")\
    .load("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower")
df.show()

## 6. More to come (stay motivated)....